# Colab · DistilRoBERTa logits for the test and out-of-scope petitions

The fine-tuned DistilRoBERTa (from `colab_gpu.ipynb`) scores the petitions that notebook 09 needs: the **6,182 test**
petitions and the **886 out-of-scope** ones. The laptop cannot run the model (no room for torch), so this short step
runs on Colab. Any runtime works; a T4 GPU takes about a minute, a CPU runtime about 10–15 minutes.

**Nothing is evaluated here.** Only `id` and `text` are read, never the labels, and no score is computed. The output
is the model's raw scores (logits). Notebook 09 applies the frozen temperature and fusion model to them and looks at
the test labels, once.

**Safety check first:** the model scores the 4,030 validation petitions again, and the logits must match the ones
saved on 1 November. This shows that the model, the tokenizer and the code are the same as before.

**Input**, on Google Drive: `petitions.parquet`, `out_of_scope.parquet`, `splits.parquet`, and from the `colab_gpu`
run `outputs/distilroberta/best_model/` and `outputs/distilroberta/val_predictions.parquet`.

**Output**, in `OUTPUT_DIR` on Drive:

| File | Content | Copy to (laptop) |
|---|---|---|
| `distilroberta_test_logits.parquet` | `id` + `logit_<class>` for the 6,182 test petitions | `data/processed/predictions/` |
| `distilroberta_out_of_scope_logits.parquet` | the same for the 886 out-of-scope petitions | `data/processed/predictions/` |
| `environment.json` | versions, runtime, the validation check | `reports/colab_test_environment.json` |

## 1. Settings

In [1]:
DRIVE_DIR = "/content/drive/MyDrive/capstone"                 # Google Drive folder with the parquet files
MODEL_DIR = f"{DRIVE_DIR}/outputs/distilroberta/best_model"     # saved by colab_gpu.ipynb on 1 Nov
VAL_PREDICTIONS = f"{DRIVE_DIR}/outputs/distilroberta/val_predictions.parquet"
OUTPUT_DIR = f"{DRIVE_DIR}/outputs/test_logits"
REPO_URL = "https://github.com/YasithaNV2001/Petition-Moderation-Assistant-CS5998-Capstone-Project.git"
REPO_DIR = "/content/repo"
MAX_LEN = 256         # the same as in training
BATCH_SIZE = 64

## 2. Google Drive, code and libraries

In [2]:
import json, os, platform, subprocess, sys, time
import numpy as np
import pandas as pd
import torch
import transformers
from google.colab import drive

drive.mount("/content/drive")
for path in [f"{DRIVE_DIR}/petitions.parquet", f"{DRIVE_DIR}/out_of_scope.parquet", f"{DRIVE_DIR}/splits.parquet",
             f"{MODEL_DIR}/config.json", f"{MODEL_DIR}/model.safetensors", VAL_PREDICTIONS]:
    assert os.path.exists(path), f"Not found on Google Drive: {path}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
REPO_COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"],
                             capture_output=True, text=True, check=True).stdout.strip()
sys.path.insert(0, f"{REPO_DIR}/src")
from preprocess import CLASSES, MODEL_INPUTS, leakage_check

GPU = torch.cuda.is_available()
DEVICE = "cuda" if GPU else "cpu"
VERSIONS = {"python": platform.python_version(), "torch": torch.__version__, "transformers": transformers.__version__,
            "pandas": pd.__version__, "numpy": np.__version__}
print("code from GitHub commit", REPO_COMMIT, "| runtime:", torch.cuda.get_device_name(0) if GPU else "CPU")
print(VERSIONS)

Mounted at /content/drive
code from GitHub commit 32c4221 | runtime: Tesla T4
{'python': '3.13.16', 'torch': '2.14.1+cu130', 'transformers': '5.22.1', 'pandas': '2.2.3', 'numpy': '2.4.6'}


## 3. Load the texts (no labels)

Only `id` and `text` are read; the `label` column is never loaded in this notebook.

In [3]:
petitions = pd.read_parquet(f"{DRIVE_DIR}/petitions.parquet", columns=["id", "text"])
petitions = petitions.merge(pd.read_parquet(f"{DRIVE_DIR}/splits.parquet"), on="id", validate="one_to_one")
val = petitions[petitions["split"] == "val"]
test = petitions[petitions["split"] == "test"]
out_of_scope = pd.read_parquet(f"{DRIVE_DIR}/out_of_scope.parquet", columns=["id", "text"])
assert (len(val), len(test), len(out_of_scope)) == (4_030, 6_182, 886)
for frame in [val, test, out_of_scope]:
    leakage_check(frame[MODEL_INPUTS])        # the model sees only the text
print(f"validation {len(val):,} | test {len(test):,} | out of scope {len(out_of_scope):,}")

validation 4,030 | test 6,182 | out of scope 886


## 4. The saved model, checked on validation

The same scoring loop as in `colab_gpu.ipynb` (section 5.5): batches of 64 texts, cut at 256 tokens, float32.
If the validation logits match the saved ones, the model and the code are unchanged. Tiny differences (about
0.0001) are normal between a GPU and a CPU; the predicted classes must be the same.

In [4]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(DEVICE)
assert [model.config.id2label[i] for i in range(len(CLASSES))] == CLASSES

def predict_logits(texts, batch_size=BATCH_SIZE):
    """Logits (n_texts x 7, in CLASSES order) for a list of texts."""
    model.eval()
    logits = []
    with torch.no_grad():
        for start in range(0, len(texts), batch_size):
            batch = tokenizer(texts[start:start + batch_size], truncation=True, max_length=MAX_LEN,
                              padding=True, return_tensors="pt").to(DEVICE)
            logits.append(model(**batch).logits.float().cpu().numpy())
    return np.vstack(logits)

In [5]:
start = time.time()
val_logits = predict_logits(val["text"].tolist())
saved = pd.read_parquet(VAL_PREDICTIONS).set_index("id").loc[val["id"]]
saved_logits = saved[[f"logit_{c}" for c in CLASSES]].to_numpy()
MAX_DIFF = float(np.abs(val_logits - saved_logits).max())
SAME_CLASS = float((val_logits.argmax(axis=1) == saved_logits.argmax(axis=1)).mean())
print(f"validation check: largest logit difference {MAX_DIFF:.6f}, same predicted class for {SAME_CLASS:.2%} "
      f"({time.time() - start:.0f} s)")
assert MAX_DIFF < 1e-3 and SAME_CLASS == 1.0, "the model does not reproduce the saved validation logits"

validation check: largest logit difference 0.000000, same predicted class for 100.00% (27 s)


## 5. Test and out-of-scope logits

Only the raw scores are saved, one row per petition, in `CLASSES` order. No probabilities and no scores here.

In [6]:
TIMINGS = {}
for name, frame in [("test", test), ("out_of_scope", out_of_scope)]:
    start = time.time()
    logits = predict_logits(frame["text"].tolist())
    table = pd.DataFrame({"id": frame["id"].to_numpy()})
    for i, c in enumerate(CLASSES):
        table[f"logit_{c}"] = logits[:, i]
    assert table["id"].is_unique and np.isfinite(logits).all()
    table.to_parquet(f"{OUTPUT_DIR}/distilroberta_{name}_logits.parquet", index=False)
    TIMINGS[name] = round((time.time() - start) / 60, 1)
    print(f"{name}: {table.shape}, {TIMINGS[name]} min")

test: (6182, 8), 0.6 min
out_of_scope: (886, 8), 0.1 min


## 6. Record and files to download

In [7]:
environment = {
    "notebook": "colab_test_logits",
    "date_utc": pd.Timestamp.now(tz="UTC").isoformat(timespec="seconds"),
    "runtime": torch.cuda.get_device_name(0) if GPU else "CPU",
    "versions": VERSIONS,
    "code_commit": REPO_COMMIT,
    "rows": {"test": len(test), "out_of_scope": len(out_of_scope)},
    "validation_check": {"max_logit_difference": MAX_DIFF, "same_predicted_class": SAME_CLASS},
    "minutes": TIMINGS,
}
with open(f"{OUTPUT_DIR}/environment.json", "w") as f:
    json.dump(environment, f, indent=2)
print(json.dumps(environment, indent=2))
for name in sorted(os.listdir(OUTPUT_DIR)):
    print(f"{name:45s} {os.path.getsize(f'{OUTPUT_DIR}/{name}') / 1e6:6.2f} MB")

{
  "notebook": "colab_test_logits",
  "date_utc": "2026-11-08T16:59:58+00:00",
  "runtime": "Tesla T4",
  "versions": {
    "python": "3.13.16",
    "torch": "2.14.1+cu130",
    "transformers": "5.22.1",
    "pandas": "2.2.3",
    "numpy": "2.4.6"
  },
  "code_commit": "32c4221",
  "rows": {
    "test": 6182,
    "out_of_scope": 886
  },
  "validation_check": {
    "max_logit_difference": 0.0,
    "same_predicted_class": 1.0
  },
  "minutes": {
    "test": 0.6,
    "out_of_scope": 0.1
  }
}
distilroberta_out_of_scope_logits.parquet       0.04 MB
distilroberta_test_logits.parquet               0.28 MB
environment.json                                0.00 MB


Download the `outputs/test_logits` folder from Drive, and save this notebook with its outputs
(File → Download → .ipynb) as `notebooks/colab_test_logits.ipynb`.